<a href="https://colab.research.google.com/github/fhzhkunming/ST-554-HW6/blob/main/HW6_Hui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ST 554 HW#6

Hui Fang

2/28/2026

# Part I - More Practice Querying a Database (16 pts)

There is a database file on the assignment link called `lahman_1871-2022.sqlite` that is an sqlite database
[downloaded from here](https://github.com/jknecht/baseball-archive-sqlite). This database has information on Major League Baseball.

1. Connect to the database and then look at all of the tables in the database (use `read_sql()` from `pandas` to have this returned as a data frame). (2 pts)

In [8]:
# import module needed
import pandas as pd
import sqlite3

# connect to the database
con = sqlite3.connect('lahman_1871-2022.sqlite')

# SQL query to list all tables
get_table = '''
    SELECT *
    FROM sqlite_schema
    WHERE type='table';
'''
# Return as a dataframe
table_df = pd.read_sql(get_table, con)
table_df

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


2. Using `SQL`, construct a table of hall of fame pitchers (any hall of famer that pitched) that gives the `playerID` and their total (sum) for `GS`, `G`, `W,` `L`, `IPOuts`, `CG`, `SHO`, and `SV` columns. The summing can be done in `pandas` or in the `SQL` call. (6 pts)

In [28]:
# set the SQL query
get_sum = '''
    SELECT p.playerID, p.GS, p.G, p.W, p.L, p.IPOuts, p.CG, p.SHO, p.SV
    FROM Pitching AS p
    JOIN HallOfFame AS h
        ON p.playerID = h.playerID
    WHERE inducted = 'Y' ;
'''
# converto to a dataframe
sum_df = pd.read_sql(get_sum, con)
# group by 'playerID' to compute sum of those variables
sum_df = sum_df.groupby('playerID').sum()
sum_df

,GS,G,W,L,IPouts,CG,SHO,SV
playerID,,,,,,,,
alexape01,599,696,373,208,15570,437,90,32
ansonca01,0,3,0,1,12,0,0,1
becklja01,1,1,0,1,12,0,0,0
bendech01,334,459,212,127,9051,255,40,34
blylebe01,685,692,287,250,14910,242,60,0
...,...,...,...,...,...,...,...,...
willivi01,471,513,249,205,11988,388,50,11
wrighge01,0,3,0,1,15,0,0,0
wrighha01,8,36,4,4,301,0,0,14


3. For all of the hall of fame pitchers, use `SQL` to create a table of their batting statistics. Namely, the `playerID` and their total (sum) for `AB`, `R`, `H`, `HR`, `RBI`, `BB`, and `SO`. The summing can be done in `pandas` or in the `SQL` call. (4 pts)

In [26]:
# set SQL qurey
get_bat = '''
    SELECT b.playerID, b.AB, b.R, b.H, b.HR, b.RBI, b.BB, b.SO
    FROM Batting AS b
    JOIN Pitching AS p
        ON b.playerID = p.playerID
    JOIN HallOfFame AS h
        ON b.playerID = h.playerID
    WHERE inducted = 'Y';
'''
# convert to a data frame
bat_df = pd.read_sql(get_bat, con)
# group by 'playerID' to compute sum of target variables
bat_df = bat_df.groupby('playerID').sum()
bat_df

,AB,R,H,HR,RBI,BB,SO
playerID,,,,,,,
alexape01,38010,3234,7938,231,3423.0,1617,5796.0
ansonca01,20562,3998,6870,194,4150.0,1968,660.0
becklja01,9551,1603,2938,87,1581.0,616,526.0
bendech01,18352,1632,3888,96,1856.0,1200,2288.0
blylebe01,10824,456,1416,0,600.0,120,4632.0
...,...,...,...,...,...,...,...
willivi01,19409,1391,3224,13,1092.0,1053,2587.0
wrighge01,5746,1330,1732,22,652.0,136,238.0
wrighha01,3252,732,896,16,452.0,148,56.0


4. Using `pandas` join the previous two tables together by pitcher. (If you want, try to do all of this via
`SQL`! Not required though, feel free to use `pd.merge()` if you’d like) (4 pts)

In [27]:
pd.merge(sum_df, bat_df, on='playerID')


,GS,G,W,L,IPouts,CG,SHO,SV,AB,R,H,HR,RBI,BB,SO
playerID,,,,,,,,,,,,,,,
alexape01,599,696,373,208,15570,437,90,32,38010,3234,7938,231,3423.0,1617,5796.0
ansonca01,0,3,0,1,12,0,0,1,20562,3998,6870,194,4150.0,1968,660.0
becklja01,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581.0,616,526.0
bendech01,334,459,212,127,9051,255,40,34,18352,1632,3888,96,1856.0,1200,2288.0
blylebe01,685,692,287,250,14910,242,60,0,10824,456,1416,0,600.0,120,4632.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
willivi01,471,513,249,205,11988,388,50,11,19409,1391,3224,13,1092.0,1053,2587.0
wrighge01,0,3,0,1,15,0,0,0,5746,1330,1732,22,652.0,136,238.0
wrighha01,8,36,4,4,301,0,0,14,3252,732,896,16,452.0,148,56.0
